## Assignment 1 - Alpha algorithm

##### Input - list of tuples of str's: [(str1, ..., strL), (strJ,...strK), ..., (strM,...,strN)]
##### Output - set of tuples of tuples str's {((str1,...,strI), (strL,...,strK)),...}

In [ ]:
from petri_net import PetriNet
from subprocess import check_call

In [2]:
log = [('a', 'b', 'c', 'd'), ('a', 'c', 'b', 'd'), ('a', 'e', 'd')]

#### Step 1. Make set of events

In [3]:
#expected output: set of str's
def all_events(log):
    tl = set()
    for item in log:
        for i in item:
            tl.add(i)
    return tl

events = all_events(log)

events

{'a', 'b', 'c', 'd', 'e'}

#### Step 2. Get all start events

In [4]:
#expected output: set of str's
def get_start_nodes(log):
    tl = set()
    for item in log:
        tl.add(item[0])
    return tl

start_nodes = get_start_nodes(log)

start_nodes

{'a'}

In [5]:
#### Step 3. Get all end events

In [6]:
#expected output: set of str's
def get_end_nodes(log):
    tl = set()
    for item in log:
        tl.add(item[-1])
    return tl

end_nodes = get_end_nodes(log)

end_nodes

{'d'}

#### Step 4. Make dataset

1. Define functions for causality, parallel and unrelated pairs<br>
2. Create footprint matrix<br>
3. Create pairs of events<br>
4. Iterate until no non-maximal pairs available

In [9]:
def get_footprint(log, events):
    direct_succession = set()
    for item in log:
        for i in range(len(item)-1):
            direct_succession.add((item[i], item[i+1]))
    
    footprint = {}

    for a in events:
        for b in events:
            if a == b:
                continue
            a_follows_b = (b, a) in direct_succession
            b_follows_a = (a, b) in direct_succession
            if b_follows_a and not a_follows_b:
                footprint[(a,b)] = {'->'}
            elif a_follows_b and not b_follows_a:
                footprint[(b,a)] = {'->'}
            elif b_follows_a and a_follows_b:
                footprint[(a,b)] = {'||'}
                footprint[(b,a)] = {'||'}
            else:
                footprint[(a,b)] = {'#'}
                footprint[(b,a)] = {'#'}

    causal = set()
    parallel = set()
    unrelated = set()

    for (a,b), rel in footprint.items():
        if '->' in rel:
            causal.add((a,b))
        elif '||' in rel:
            parallel.add((a,b))
        elif '#' in rel:
            unrelated.add((a,b))

    return footprint, causal, parallel, unrelated

In [10]:
footprint, causal, parallel, unrelated = get_footprint(log, events)

In [11]:
print(f"len(causal): {len(causal)}")
print(f"len(parallel): {len(parallel)}")
print(f"len(unrelated): {len(unrelated)}")

len(causal): 6
len(parallel): 2
len(unrelated): 6


In [12]:
import pandas as pd
footprint_matrix = pd.DataFrame(index=list(events), columns=list(events), dtype=str)
for a in events:
    for b in events:
        if a == b:
            footprint_matrix.loc[a,b] = '-'
        elif (a,b) in causal:
            footprint_matrix.loc[a,b] = '→'
            footprint_matrix.loc[b,a] = '←'
        elif (a,b) in parallel: 
            footprint_matrix.loc[a,b] = '||'
        elif (a,b) in unrelated:
            footprint_matrix.loc[a,b] = '#' 
display(footprint_matrix)

,a,b,e,c,d
a,-,→,→,→,#
b,←,-,#,||,→
e,←,#,-,#,→
c,←,||,#,-,→
d,#,←,←,←,-


In [13]:
import itertools

def all_causal(A, B, causal):
    for a in A:
        for b in B:
            if (a,b) not in causal:
                return False
    return True

def all_unrelated(S, unrelated):
    for x,y in itertools.combinations(S, 2):
        if (x,y) not in unrelated:
            return False
    return True

def is_maximal(A, B, events, causal, unrelated):
    for x in events - A:
        new_A = A.union({x})
        if all_causal(new_A, B, causal) and all_unrelated(new_A, unrelated):
            return False
    for y in events - B:
        new_B = B.union({y})
        if all_causal(A, new_B, causal) and all_unrelated(new_B, unrelated):
            return False
    return True

In [14]:
def get_places(events, start_nodes, end_nodes, causal, unrelated):
    max_set_size = len(events) # может потребоваться ограничить при слишком большом len(events)/размере данных; тут отрабатывает быстро

    subsets = []
    for k in range(1, max_set_size+1):
        for comb in itertools.combinations(events, k):
            subsets.append(set(comb))

    places = set()
    for A in subsets:
        for B in subsets:
            if not all_causal(A, B, causal):
                continue
            if not all_unrelated(A, unrelated):
                continue
            if not all_unrelated(B, unrelated):
                continue
            if is_maximal(A, B, events, causal, unrelated):
                places.add((frozenset(A), frozenset(B)))

    start_candidate = (frozenset(), frozenset(start_nodes))
    if all_unrelated(start_nodes, unrelated):
        maximal_start = True
        for y in events - start_nodes:
            new_B = start_nodes.union({y})
            if all_unrelated(new_B, unrelated):
                maximal_start = False
                break
        if maximal_start:
            places.add(start_candidate)

    end_candidate = (frozenset(end_nodes), frozenset())
    if all_unrelated(end_nodes, unrelated):
        maximal_end = True
        for x in events - end_nodes:
            new_A = end_nodes.union({x})
            if all_unrelated(new_A, unrelated):
                maximal_end = False
                break
        if maximal_end:
            places.add(end_candidate)

    return places


In [15]:
def get_dataset(places):
    dataset = set()
    for A, B in places:
        tuple_A = tuple(sorted(A))
        tuple_B = tuple(sorted(B))
        dataset.add((tuple_A, tuple_B))
    return dataset

In [16]:
dataset = get_dataset(get_places(events, start_nodes, end_nodes, causal, unrelated))
dataset

{(('a',), ('b', 'e')),
 (('a',), ('c', 'e')),
 (('b', 'e'), ('d',)),
 (('c', 'e'), ('d',))}

Expected output (order irrelevant): {(('a',), ('c', 'e')),
 (('a',), ('e', 'b')),
 (('c', 'e'), ('d',)),
 (('e', 'b'), ('d',))} 

#### Let's check the result with PetriNet drawer

In [17]:
pn = PetriNet()
filename = 'my_first_alpha_miner'
pn.generate_with_alpha(tl=events,
                       ti=start_nodes,
                       to=end_nodes,
                       yl=dataset,
                       dotfile="{}.dot".format(filename))
check_call(["dot", "-Tpng", "{}.dot".format(filename),"-o", "{}.png".format(filename)])

0

## Another log for testing

Input:

In [18]:
log = [('a', 'b', 'e', 'f'),
 ('a', 'b', 'e', 'c', 'd', 'b', 'f'),
 ('a', 'b', 'c', 'e', 'd', 'b', 'f'),
 ('a', 'b', 'c', 'd', 'e', 'b', 'f'),
 ('a', 'e', 'b', 'c', 'd', 'b', 'f')]
log

[('a', 'b', 'e', 'f'),
 ('a', 'b', 'e', 'c', 'd', 'b', 'f'),
 ('a', 'b', 'c', 'e', 'd', 'b', 'f'),
 ('a', 'b', 'c', 'd', 'e', 'b', 'f'),
 ('a', 'e', 'b', 'c', 'd', 'b', 'f')]

Expected output:

In [19]:
dataset = {(('a',), ('e',)), (('a', 'd'), ('b',)), (('b',), ('c', 'f')), (('c',), ('d',)), (('e',), ('f',))}
dataset

{(('a',), ('e',)),
 (('a', 'd'), ('b',)),
 (('b',), ('c', 'f')),
 (('c',), ('d',)),
 (('e',), ('f',))}

In [21]:
events = all_events(log)
start_nodes = get_start_nodes(log)
end_nodes = get_end_nodes(log)

footprint, causal, parallel, unrelated = get_footprint(log, events)
dataset = get_dataset(get_places(events, start_nodes, end_nodes, causal, unrelated))
dataset

{(('a',), ('e',)),
 (('a', 'd'), ('b',)),
 (('b',), ('c', 'f')),
 (('c',), ('d',)),
 (('e',), ('f',))}